# Caldis — First Example: source → valve → sink

The simplest possible Caldis system: water flows from a **source**, through a
**valve** (a pressure drop), into a **sink**. No heat exchange, no loop — just to
see the *build → connect → impose → solve → read* loop end to end.

The idea behind Caldis: a component does not compute outputs, it declares the
**equations** its physics must satisfy. Here the valve declares three:
mass conservation, its flow-vs-pressure-drop law, and constant enthalpy
(a valve neither heats nor cools). The solver finds the state that satisfies them.

In [1]:
# --- imports ---
import numpy as np
from CoolProp.CoolProp import PropsSI

import caldis
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Valve
from caldis.fluids import CoolPropBackend
from caldis.solvers import solve

print("Caldis version:", getattr(caldis, "__version__", "unknown"))

Caldis version: 0.2.0


## Building the system

Three pieces:
- **`src`** — a `FluidBoundary` imposing the inlet pressure and temperature. It does
  NOT impose the flow: the valve law will set it. (Sign convention: `mdot > 0` enters
  a component.)
- **`valve`** — a `Valve` with a flow coefficient `Kv`: it produces a mass flow
  `mdot = Kv * sqrt(dp)` and conserves enthalpy.
- **`snk`** — a `FluidBoundary` imposing the outlet pressure.

The pressure difference between `src` (3 bar) and `snk` (1 bar) drives the flow; the
valve law turns that Δp into a mass flow. Nothing is over-imposed, so the system is
square and the flow is a *result*.

In [2]:
be = CoolPropBackend("Water")

p_in, T_in = 3.0e5, 40 + 273.15     # 3 bar, 40 C
p_out      = 1.0e5                  # 1 bar

src   = FluidBoundary("src", p=p_in, T=T_in, backend=be)   # flow left open
valve = Valve("valve", Kv=1.0e-4, backend=be)
snk   = FluidBoundary("snk", p=p_out, backend=be)

sys_ = System("valve_demo")
sys_.add(src, valve, snk)
sys_.connect(src.port,  valve.inl)
sys_.connect(valve.out, snk.port)

Connection(valve.out <-> snk.port)

## Solving

`solve()` runs the whole workflow: it checks the system is square (DOF), that the
start point is evaluable, warns about any physical issue, then solves. On success it
prints `✓ SOLVED`; on failure it prints diagnostics naming what went wrong.

This system is simple enough to need no start hints — the fluid-aware defaults are
good enough.

In [3]:
sol = solve(sys_)

✓ DOF: square (12 unknowns)
✓ start point: evaluable
✓ SOLVED (|F_scaled|=2.3e-10)


## Reading the results

Each port exposes its solved state on demand: `port.p`, `port.mdot`, `port.h`, plus
`port.T` (K), `port.T_C` (°C), `port.x` (quality). `system.port_table()` shows every
port at once.

Check the physics: the valve is **isenthalpic**, so the outlet temperature should be
almost identical to the inlet (a tiny change only, from the pressure drop). The mass
flow should equal `Kv * sqrt(p_in - p_out)`.

In [4]:
m = valve.inl.mdot
print(f"inlet    : {valve.inl.T_C:.3f} °C   {valve.inl.p/1e5:.2f} bar")
print(f"outlet   : {valve.out.T_C:.3f} °C   {valve.out.p/1e5:.2f} bar")
print(f"mass flow: {m*1e3:.3f} g/s")

# cross-check against the valve law  mdot = Kv * sqrt(dp)
Kv = valve.Kv                       # output (value), not valve.params["Kv"]
dp = p_in - p_out
print(f"expected : {Kv*np.sqrt(dp)*1e3:.3f} g/s   (Kv * sqrt(dp))")

sys_.port_table()

inlet    : 40.000 °C   3.00 bar
outlet   : 40.042 °C   1.00 bar
mass flow: 44.721 g/s
expected : 44.721 g/s   (Kv * sqrt(dp))


p_bar           T        T_C      h_kJkg   s_kJkgK    x  \
component port                                                            
src       port    3.0  313.150000  40.000000  167.792339  0.572288 -1.0   
valve     inl     3.0  313.150000  40.000000  167.792339  0.572288 -1.0   
          out     1.0  313.192405  40.042405  167.792339  0.572932 -1.0   
snk       port    1.0  313.192405  40.042405  167.792339  0.572932 -1.0   

                       rho      mdot  
component port                        
src       port  992.303539 -0.044721  
valve     inl   992.303539  0.044721  
          out   992.199546 -0.044721  
snk       port  992.199546  0.044721

## What to try next

- Change `p_out` and re-run: the flow follows `sqrt(dp)`, the temperature barely moves.
- Call `valve.structure()` to see its ports and variables, or `valve.describe()` for
  its values after the solve.
- Replace the valve by another component (a heat exchanger, a pump) and reconnect —
  the build/solve/read loop stays the same.

Then move on to `01_getting_started.ipynb` (a compressor) and the full heat-pump and
gas-turbine examples.